In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "text"])
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


In [ ]:
df.shape

(5572, 2)

In [ ]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [ ]:
df.isnull().sum()

,0
label,0
text,0


In [ ]:
df["label"].value_counts(normalize=True)

,proportion
label,
0,0.865937
1,0.134063


In [ ]:
df["text"].head()

,text
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df["text"]=df["text"].str.lower()

In [ ]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans("","",string.punctuation))
df["text"]=df["text"].apply(remove_punc)

In [ ]:
def remove_num(txt):
  new=""
  for i in txt:
    if not i.isdigit():
        new+=i
  return new
df["text"]=df["text"].apply(remove_num)

In [ ]:
def remove_extra(txt):
  new=""
  for i in txt:
    if i.isascii():
      new+=i
  return new
df["text"]=df["text"].apply(remove_extra)

In [ ]:
df["text"] = df["text"].str.strip()

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words=set(stopwords.words("english"))
def remove_stopwords(text):
    words = text.split()
    return " ".join([w for w in words if w not in stop_words])

df["text"] = df["text"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
df["text"].head(10)

,text
0,go jurong point crazy available bugis n great ...
1,ok lar joking wif u oni
2,free entry wkly comp win fa cup final tkts st ...
3,u dun say early hor u c already say
4,nah dont think goes usf lives around though
5,freemsg hey darling weeks word back id like fu...
6,even brother like speak treat like aids patent
7,per request melle melle oru minnaminunginte nu...
8,winner valued network customer selected receiv...
9,mobile months u r entitled update latest colou...


In [ ]:
from sklearn.model_selection import train_test_split
X=df["text"]
y=df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer_tfidf= TfidfVectorizer()
x_train_tfidf=vectorizer_tfidf.fit_transform(X_train)
x_test_tfidf=vectorizer_tfidf.transform(X_test)

In [ ]:
x_train_tfidf.shape

(3733, 6800)

In [ ]:
x_test_tfidf.shape

(1839, 6800)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr= LogisticRegression()
model_lr.fit(x_train_tfidf,y_train)

LogisticRegression()

In [ ]:
y_pred_lr= model_lr.predict(x_test_tfidf)

In [ ]:
from sklearn.metrics import accuracy_score,f1_score
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1 score:", f1_score(y_test, y_pred_lr))

Accuracy: 0.9603045133224578
F1 score: 0.8298368298368298


In [ ]:
from sklearn.naive_bayes import MultinomialNB
model_nb_=MultinomialNB()
model_nb_.fit(x_train_tfidf,y_train)

MultinomialNB()

In [ ]:
y_pred_nb=model_nb_.predict(x_test_tfidf)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("F1 score:", f1_score(y_test, y_pred_nb))

Accuracy: 0.9662860250135944
F1 score: 0.8558139534883721


In [ ]:
import joblib

joblib.dump(model_nb_, "email_model.pkl")
joblib.dump(vectorizer_tfidf, "tfidf.pkl")

['tfidf.pkl']

In [ ]:
from google.colab import files

files.download("email_model.pkl")
files.download("tfidf.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>